# 08 · Vector Databases

**Idea:** the NumPy store re-embeds everything each run and lives in memory. A real **vector database** persists the index to disk and offers metadata filters and fast approximate search. Here we use **Chroma** with the same `search` surface.

> Needs `pip install chromadb`.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'rag_lab').exists())
sys.path.insert(0, str(root))

import tempfile, pathlib
from rag_lab import load_markdown, ChromaStore
persist = str(pathlib.Path(tempfile.gettempdir()) / 'rag_chroma_demo')
store = ChromaStore(collection='notes', persist_dir=persist)
if len(store) == 0:
    store.add(load_markdown())
print('persisted at:', persist, '| vectors:', len(store))

In [ ]:
for h in store.search('what is reciprocal rank fusion?', k=3):
    print(f'{h.score:.3f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

Reopen the same directory in a fresh store: the index is already there — no re-embedding. That persistence is the whole point of a vector DB.

In [ ]:
reopened = ChromaStore(collection='notes', persist_dir=persist)
print('reopened vectors (no re-embed):', len(reopened))

Metadata filters run *inside* the database. Restrict to one source file, then rank.

In [ ]:
hits = store.search('chunk boundaries', k=3, where={'source': 'chunking_strategies.md'})
for h in hits:
    print(f'{h.score:.3f}  {h.chunk.source} :: {h.chunk.meta.get("heading")}')

**Takeaway:** a vector DB adds persistence, metadata filtering, and approximate search (HNSW under the hood — notebook 24) behind the same retrieval interface.

Next → `09_keyword_search.ipynb`